# RSNA Knee Abnormality Detection — detailed baseline (v2.6)

Predicts per-study probabilities for **12 knee abnormalities** from knee MRI
DICOM series + radiology report text. Evaluation: macro average ROC-AUC.

Designed for the competition constraints:

- total notebook runtime <= 9 h (GPU recommended; CPU fallback config included,
  plus an automatic runtime budget guard)
- **internet DISABLED at scoring** -> torch is installed from an attached
  wheel dataset (`nururrr/rsna-torch-wheels-cu126`, torch 2.7.1+cu126 for
  Pascal GPUs) and pretrained weights are
  attached as a dataset (`nururrr/rsna-backbone`, dumped via `CFG.save_weights`)
- all attached datasets must be **publicly available** (competition rule)
- output file must be `submission.csv` with the exact header

> This notebook was verified end-to-end on the real competition data (dev
> runs on Kaggle GPUs): final config 3-fold DenseNet121 + report text on the
> 58 fully-labeled studies, val macro AUC 0.5256 (fold AUCs 0.486/0.489/0.603,
> weak labels dominate the signal). v2.5 added low-weight pseudo labels mined
> from radiology reports of unlabeled studies (~76% agreement on the labeled
> subset, 500 extra studies); v2.6 makes the environment **scoring-compliant**
> (internet disabled: offline torch wheels + CPU fallback instead of crash).

## What is inside (why)

| Step | Choice | Rationale |
|---|---|---|
| Series selection | Sagittal + Coronal, fluid-sensitive first | ACL/MCL/menisci best on sagittal; coronal adds MCL/OA signal; fluid-sensitive (T2/PD/STIR) shows effusion, synovitis, contusion |
| DICOM decoding | VOI LUT, MONOCHROME1 fix, rescale slope/intercept, percentile windowing | correct intensity semantics across 16 sites |
| Physical resampling | resample by PixelSpacing to ~0.5 mm/px | normalizes anatomy scale across scanners/sites |
| Disk cache | uint8 `.npy` per series | decode once, epoch loop becomes I/O-light |
| Patient-aware CV | StratifiedGroupKFold on patient_id | prevents leakage when a patient has several studies |
| Slice aggregation | attention pooling over valid slices | learn which slices matter instead of blind mean |
| Text branch | char n-gram hashing (stateless, offline) + MLP head | uses the paired radiology report, multilingual by construction, zero pretrained dependency |
| Pseudo labels | report-keyword weak labels on unlabeled studies (weight 0.3, NaN targets masked) | only ~58/4407 studies have full labels; report text agrees with true labels ~76% |
| Training | EMA, warmup+cosine, grad clip, label smoothing, AMP, early stop | stable, better final weights |
| Inference | sliding-window TTA + fold ensemble | robust probabilities |
| Runtime budget | stop adding folds if > 8.5 h elapsed | never breach the 9 h limit |
| Offline torch | install 2.4.1+cu118 from attached wheel dataset (`--no-index`), PyPI only as dev fallback | scoring has internet disabled; preinstalled cu128 torch lacks kernels on P100/T4-era GPUs |

Run order:  config -> data -> pseudo labels -> EDA -> CV splits -> series
selection -> preprocessing + cache -> text features -> dataset -> model ->
train folds -> inference -> submission


In [ ]:
import gc, glob, json, math, os, random, re, time, warnings
from collections import Counter

# ---- GPU check + torch reinstall (OFFLINE-SAFE) ----
# Scoring runs with internet DISABLED, so prefer installing torch from the
# attached wheel dataset (rsna-torch-wheels-cu126) with --no-index; the PyPI
# fallback only works during development. Why: the preinstalled torch
# 2.10.0+cu128 has no CUDA kernels for older GPUs (e.g. P100 sm_6.0 ->
# "no kernel image is available"; PyTorch >= 2.8 dropped sm_50-sm_60 for
# CUDA 12.8+ builds), so we downgrade to 2.7.1+cu126 - the last build with
# Pascal kernels. Install is --no-deps: the cu126 wheel relies on the pip
# nvidia-cu12 runtime libs the kernel already ships with preinstalled torch
# (checked by _nvidia_libs_ready before installing).
import subprocess, sys

TORCH_VER, TORCHVISION_VER = "2.7.1+cu126", "0.22.1+cu126"

def install_torch(use_index):
    # Returns True on success. use_index=False -> local wheels only.
    if use_index:
        args = [sys.executable, "-m", "pip", "install", "-q",
                f"torch=={TORCH_VER}", f"torchvision=={TORCHVISION_VER}",
                "--index-url", "https://download.pytorch.org/whl/cu126"]
    else:
        # kaggle auto-extracts the uploaded wheels_cu126.zip; the dataset
        # ships torch/torchvision 2.7.1+cu126 AND the exact nvidia-cu12
        # dependency wheels (env images drop some of their libs, e.g.
        # libcusparseLt.so.0), so --no-deps install of all of them is safe.
        py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
        whls = sorted(glob.glob(
            "/kaggle/input/**/wheels_cu126/*.whl", recursive=True))
        whls = [w for w in whls
                if w.split("/")[-1].startswith(("torch-", "torchvision-",
                                                "nvidia_"))
                and (f"-{py_tag}-" in w or "py3-none-" in w)]
        if not whls:
            print("  (no wheels found under wheels_cu126/ in attached datasets)")
            return False
        args = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                "--no-deps", *whls]
    return subprocess.run(args).returncode == 0

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True)
reinstalled = False
if gpu.returncode == 0 and gpu.stdout.strip():
    name_s, cap_s = gpu.stdout.strip().splitlines()[0].rsplit(",", 1)
    cap = tuple(int(x) for x in cap_s.strip().split("."))
    print(f"GPU: {name_s.strip()} | compute capability: {cap[0]}.{cap[1]}")
    if cap[0] < 8:
        if install_torch(use_index=False):
            print(f"installed torch {TORCH_VER} from attached wheel dataset")
            reinstalled = True
        elif install_torch(use_index=True):
            print(f"installed torch {TORCH_VER} from PyPI (needs internet)")
            reinstalled = True
        else:
            print("WARNING: torch reinstall failed - continuing with preinstalled torch")
else:
    print("GPU: nvidia-smi unavailable")

# after a reinstall the kernel's LD_LIBRARY_PATH may miss some nvidia-cu12
# lib dirs (e.g. libcusparseLt.so.0 from nvidia-cusparselt-cu12), so expose
# them all before importing torch
if reinstalled:
    nvdirs = sorted(glob.glob(
        "/usr/local/lib/python3.12/dist-packages/nvidia/*/lib"))
    if nvdirs:
        os.environ["LD_LIBRARY_PATH"] = ":".join(
            nvdirs + [os.environ.get("LD_LIBRARY_PATH", "")])
        print(f"LD_LIBRARY_PATH extended with {len(nvdirs)} nvidia lib dirs")
    # some kernel images also drop the .so soname symlinks of the nvidia-cu12
    # packages (e.g. libcusparseLt.so.0); recreate them from the real files
    made = 0
    for d in nvdirs:
        for f in sorted(glob.glob(f"{d}/*.so.*")):
            b = os.path.basename(f)
            s = b.rsplit(".", 2)[0]
            if (b.startswith("lib") and s != b
                    and not os.path.exists(os.path.join(d, s))):
                try:
                    os.symlink(b, os.path.join(d, s))
                    made += 1
                except OSError:
                    pass
    if made:
        print(f"created {made} missing nvidia .so soname symlinks")

import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, KFold

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGKF = True
except ImportError:
    HAS_SGKF = False

warnings.filterwarnings("ignore")

START_TIME = time.time()
SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    try:
        x = torch.ones(4, device=DEVICE)
        (x + x).cpu()
        torch.cuda.synchronize()
        print("CUDA kernel check: OK")
    except Exception as e:
        print(f"CUDA kernel check FAILED ({e})")
        print("-> falling back to CPU (slow; config will be shrunk)")
        DEVICE = "cpu"


class CFG:
    # ---- data ----
    # Kaggle mounts competition data either as /kaggle/input/<slug> or
    # /kaggle/input/competitions/<slug> - pick whichever is present.
    data_dir  = next((p for p in ("/kaggle/input/rsna-knee-abnormality-detection",
                                  "/kaggle/input/competitions/rsna-knee-abnormality-detection")
                      if os.path.exists(os.path.join(p, "train.csv"))),
                     "/kaggle/input/rsna-knee-abnormality-detection")
    train_dir = f"{data_dir}/train_series"
    test_dir  = f"{data_dir}/test_series"
    out_dir   = "/kaggle/working"
    cache_dir = "/kaggle/working/cache"

    # ---- model ----
    backbone        = "densenet121"          # densenet121 | efficientnet_b0
    # attached backbone weights (offline scoring): auto-detected from the
    # rsna-backbone dataset; None (random init) only if nothing is attached
    pretrained_path = next(iter(sorted(glob.glob(
        "/kaggle/input/rsna-backbone/**/*.pth", recursive=True)) or
        glob.glob("/kaggle/input/**/densenet121_backbone.pth", recursive=True) or []),
        None)
    save_weights    = False                  # DEV ONLY: dump ImageNet backbone to out_dir
    use_compile     = False                  # torch.compile (GPU, optional speedup)
    use_text        = True                   # radiology report branch

    # ---- preprocessing ----
    img_size         = 224
    target_px_mm     = 0.5                   # physical pixel target for resampling
    n_slices         = 24                    # slices sampled per series
    max_slices_cache = 48                    # slices kept in the on-disk cache
    n_series         = 2                     # series per study (1 = sagittal, 2 = + coronal)

    # ---- training ----
    folds           = 3
    epochs          = 4
    freeze_epochs   = 1                      # head-only first epoch(s)
    patience        = 2                      # early stopping on val macro AUC
    ema_decay       = 0.99
    lr_head         = 1e-3
    lr_body         = 3e-5
    weight_decay    = 1e-2
    warmup_frac     = 0.05
    grad_clip       = 1.0
    label_smoothing = 0.02
    batch_size      = 4                      # samples per step (each = n_slices images)
    frames_chunk    = 64                     # max frames per GPU conv chunk (OOM guard)
    num_workers     = 2
    use_amp         = torch.cuda.is_available()

    # ---- inference / limits ----
    tta_grids    = 3 if torch.cuda.is_available() else 1
    budget_hours = 8.5                       # skip further folds beyond this
    subset       = 1.0                       # fraction of labeled studies (quick tests)

    # ---- pseudo labels (weak labels from radiology reports) ----
    use_pseudo    = True                     # add report-keyword labels of unlabeled studies
    pseudo_max    = 500                      # max pseudo studies added
    pseudo_weight = 0.3                      # loss weight for pseudo labels


if DEVICE == "cpu":
    # CPU fallback (e.g. scoring without usable CUDA): shrink the workload so
    # a valid submission still fits the 9 h budget (weak, but not a crash).
    print("CPU mode: shrinking config (n_series=1, batch=2, folds=1, epochs=1, TTA=1)")
    CFG.n_series   = 1
    CFG.folds      = 1
    CFG.epochs     = 1
    CFG.batch_size = 2
    CFG.tta_grids  = 1
    CFG.use_amp    = False


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

os.makedirs(CFG.cache_dir, exist_ok=True)
os.makedirs(CFG.out_dir, exist_ok=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.enabled = True
torch.set_num_threads(min(os.cpu_count() or 4, 8))


def elapsed_hours():
    return (time.time() - START_TIME) / 3600.0


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"config: backbone={CFG.backbone} folds={CFG.folds} epochs={CFG.epochs} "
      f"n_series={CFG.n_series} text={CFG.use_text} tta={CFG.tta_grids}")


## 1. Data overview

`train.csv`: one row per study (`StudyInstanceUID`, `PatientSex`, free-text
`Report`, 12 binary labels). **Only a subset of training studies carries
labels** — we train on labeled rows and keep the reports for the text branch.
`train_series.csv` / `test_series.csv`: series descriptors
(`Fluid_Sensitive`, `Fat_Suppression`, `Anatomical_Plane`). DICOM slices:
`train_series/<Study>/<Series>/*.dcm`.

In [ ]:
train_df   = pd.read_csv(f"{CFG.data_dir}/train.csv")
train_ser  = pd.read_csv(f"{CFG.data_dir}/train_series.csv")
test_ser   = pd.read_csv(f"{CFG.data_dir}/test_series.csv")
sample_sub = pd.read_csv(f"{CFG.data_dir}/sample_submission.csv")

TARGETS = [c for c in sample_sub.columns if c != "StudyInstanceUID"]
assert len(TARGETS) == 12, TARGETS

labeled = train_df.dropna(subset=TARGETS).copy()
labeled = labeled.drop_duplicates(subset="StudyInstanceUID").reset_index(drop=True)

# subset MUST be applied here (before CV splits are built), otherwise the
# fold indices computed later would point at the wrong rows
if CFG.subset < 1.0:
    rng = np.random.RandomState(SEED)
    keep = rng.choice(len(labeled), max(1, int(len(labeled) * CFG.subset)), replace=False)
    labeled = labeled.iloc[keep].reset_index(drop=True)
    print(f"[subset] -> {len(labeled)} labeled studies")

print("train.csv   :", train_df.shape, "| columns:", list(train_df.columns))
print("labeled     :", len(labeled), "of", len(train_df), "studies")
print("test_series :", test_ser.shape, "| test studies:", test_ser.StudyInstanceUID.nunique())
print("targets     :", TARGETS)
print()
print("report char-length stats (train):")
print(train_df["Report"].fillna("").str.len().describe().round(0).to_string())


In [ ]:
# ---- weak labels from radiology reports (multi-language keywords) ----
# Only ~58 of 4407 train studies carry all 12 labels; the rest are all-NaN.
# Keyword rules over the Report text give noisy (but useful) labels, verified
# below against the true labels (~76% agreement). Used as low-weight
# auxiliary training targets; undecided targets are masked from the loss.
PSEUDO_RULES = {
    "ACL": (
        [r"anterior cruciate", r"\bacl\b", r"cruzado anterior",
         r"vordere[sn]? kreuzband", r"crois[ée] ant[ée]rieur", r"çapraz ba"],
        [r"intact", r"preserv", r"normal", r"unremarkable", r"without",
         r"no tear", r"not torn", r"sin rotura", r"no rotura", r"no rasg",
         r"no hay", r"nicht", r"keine", r"aucune", r"sans", r"no evidencia",
         r"no evid", r"denied", r"rule[ds]? out", r"excluded", r"no increased intensity"],
    ),
    "MCL": (
        [r"medial collateral", r"\bmcl\b", r"colateral medial",
         r"inner[es] seitenband", r"collat[ée]ral m[ée]dial"],
        [r"intact", r"preserv", r"normal", r"unremarkable", r"without",
         r"no tear", r"not torn", r"no rasg", r"no hay", r"nicht", r"keine",
         r"aucune", r"sans", r"denied", r"excluded", r"no increased intensity"],
    ),
    "Medial Meniscus": (
        [r"medial menisc", r"menisco interno", r"medial meniscal",
         r"innenmeniskus", r"m[ée]nisque interne"],
        [r"intact", r"preserv", r"normal", r"unremarkable", r"without",
         r"no tear", r"not torn", r"sin rotura", r"no rotura", r"no rasg",
         r"no hay", r"nicht", r"keine", r"aucune", r"sans", r"denied", r"excluded"],
    ),
    "Lateral Meniscus": (
        [r"lateral menisc", r"menisco externo", r"lateral meniscal",
         r"au[ss]enmeniskus", r"m[ée]nisque externe"],
        [r"intact", r"preserv", r"normal", r"unremarkable", r"without",
         r"no tear", r"not torn", r"sin rotura", r"no rotura", r"no rasg",
         r"no hay", r"nicht", r"keine", r"aucune", r"sans", r"denied", r"excluded"],
    ),
    "Medial OA": (
        [r"medial (compartment|comparti\w*|femorotibial|joint space|tibiofemoral).{0,60}(narrow|arthros|gonarthros|osteophyt|cartilage loss|chondral loss|degenerat|spur|sclerosis)",
         r"(medial|m[ée]diale) (gon)?arthros\w*", r"medial osteoarthritis",
         r"tricompartmental|all three compartments|all 3 compartments"],
        [r"normal", r"unremarkable", r"preserved", r"no narrowing",
         r"no joint space", r"no osteophyt", r"no spur",
         r"without .{0,30}(narrow|arthros)", r"no .{0,30}(arthros|narrow)",
         r"sin .{0,30}(artrosis|estrech|osteofit)", r"no .{0,30}osteoarthritis",
         r"intact", r"no .{0,30}gonarthrose", r"no hay .{0,30}artrosis"],
    ),
    "Lateral OA": (
        [r"lateral (compartment|comparti\w*|femorotibial|joint space|tibiofemoral).{0,60}(narrow|arthros|gonarthros|osteophyt|cartilage loss|chondral loss|degenerat|spur|sclerosis)",
         r"(lateral|l[ée]g[ée]re) (gon)?arthros\w*", r"lateral osteoarthritis",
         r"tricompartmental|all three compartments|all 3 compartments"],
        [r"normal", r"unremarkable", r"preserved", r"no narrowing",
         r"no joint space", r"no osteophyt", r"no spur",
         r"without .{0,30}(narrow|arthros)", r"no .{0,30}(arthros|narrow)",
         r"sin .{0,30}(artrosis|estrech|osteofit)", r"no .{0,30}osteoarthritis",
         r"intact", r"no .{0,30}gonarthrose"],
    ),
    "PF OA": (
        [r"patellofemoral.{0,60}(arthros|chondromalacia|narrow|osteophyt|cartilage loss|chondral loss|degenerat|spur|sclerosis|cliff)",
         r"tricompartmental|all three compartments|all 3 compartments",
         r"retropatell\w* (arthrose|artrose|narrow|degenerat)",
         r"(chondromalacia|cartilage loss|chondral loss|narrow).{0,40}patellofemoral"],
        [r"normal", r"unremarkable", r"congru", r"preserved", r"intact",
         r"no .{0,30}(arthros|osteoarthritis|chondr|narrow)", r"without .{0,30}(arthros|chondr|narrow)",
         r"sin .{0,30}(artrosis|condr|estrech)", r"no .{0,30}gonarthrose",
         r"no hay .{0,30}(artrosis|condr)"],
    ),
    "Effusion": (
        [r"effusion", r"derrame", r"erguss", r"[ée]panchement", r"efusi[oó]n",
         r"efüzyon", r"joint fluid", r"articular fluid", r"fluid accumulation",
         r"hidrartrosis", r"hydrarthrosis", r"effusie", r"synovial fluid"],
        [r"no .{0,25}effusion", r"no .{0,25}derrame", r"no .{0,25}erguss",
         r"no .{0,25}[ée]panchement", r"no joint fluid", r"no fluid",
         r"sin derrame", r"without .{0,25}effusion", r"dry joint",
         r"no .{0,25}efusi", r"keine .{0,25}erguss", r"nicht .{0,25}erguss"],
    ),
    "Synovitis": (
        [r"synovitis", r"sinovitis", r"synovial", r"pannus", r"sinovial"],
        [r"no .{0,25}synovitis", r"no .{0,25}sinovitis", r"no .{0,25}synovial",
         r"no synovial", r"without .{0,25}synovial", r"normal synovial",
         r"sin sinovitis", r"no sinovial", r"no .{0,25}sinovial",
         r"keine .{0,25}synovitis", r"nicht .{0,25}synovitis"],
    ),
    "Baker's": (
        [r"baker", r"popliteal cyst", r"popliteus cyst", r"poplitea cyst",
         r"quiste popl", r"poplitealzyste", r"kyste popl", r"bakers cyste"],
        [r"no .{0,25}baker", r"no .{0,25}poplit", r"sin .{0,25}quiste",
         r"no cyst", r"without .{0,25}cyst", r"no .{0,25}cyst", r"keine .{0,25}zyste",
         r"nicht .{0,25}zyste"],
    ),
    "Contusion": (
        [r"contusion", r"contusi", r"bone bruise", r"bone contusion",
         r"knochenkontusion", r"bone bruising", r"bruis[ei]ng"],
        [r"no .{0,35}(contusion|bruise|bruis)", r"without .{0,35}(contusion|bruise)",
         r"sin .{0,35}contusion", r"no .{0,25}contusion", r"no bone bruise",
         r"keine .{0,35}kontusion", r"nicht .{0,35}kontusion",
         r"no hay .{0,35}contusion"],
    ),
    "Fracture": (
        [r"fracture", r"fractura", r"\bfx\b", r"fractuur", r"fraktur", r"fractur"],
        [r"no .{0,25}fracture", r"no .{0,25}fractura", r"no acute fracture",
         r"no evidence of fracture", r"no evidencia de fractura",
         r"no .{0,25}fraktur", r"nicht .{0,25}fraktur", r"keine .{0,25}fraktur",
         r"without .{0,25}fracture", r"sin fractura", r"no .{0,25}fx\b"],
    ),
}


def report_label(text, pos_pats, neg_terms):
    if not isinstance(text, str) or not text.strip():
        return None
    t = re.sub(r"\s+", " ", text).lower()
    negs = []
    for n in neg_terms:
        for m in re.finditer(n, t):
            negs.append((m.start(), m.end()))
    pos_hit = 0
    for p in pos_pats:
        for m in re.finditer(p, t):
            if any(n_end >= m.start() - 30 and n_start <= m.end() + 40
                   for n_start, n_end in negs):
                continue
            pos_hit += 1
    if pos_hit > 0:
        return 1
    return 0 if negs else None


def make_pseudo(df, rules):
    out = {t: [] for t in rules}
    for _, r in df.iterrows():
        for t in rules:
            out[t].append(report_label(r["Report"], *rules[t]))
    p = pd.DataFrame(out)
    p.insert(0, "StudyInstanceUID", df["StudyInstanceUID"].values)
    return p


pseudo = make_pseudo(train_df, PSEUDO_RULES)
pseudo["n_dec"] = pseudo[TARGETS].notna().sum(axis=1)

if CFG.use_pseudo:
    agree = pseudo.set_index("StudyInstanceUID").loc[labeled.StudyInstanceUID][TARGETS]
    y_true = labeled.set_index("StudyInstanceUID")[TARGETS]
    both = agree.notna() & y_true.notna()
    n_dec = int(both.sum().sum())
    eq = ((agree.to_numpy() == y_true.to_numpy()) & both.to_numpy())
    print("pseudo-vs-true agreement (labeled studies):")
    print(f"  overall {eq.sum() / max(n_dec, 1):.3f}  decisions {n_dec}")
    per = {}
    for t in TARGETS:
        m = both[t]
        per[t] = (agree[t][m] == y_true[t][m]).mean() if m.any() else float("nan")
    print("  " + "  ".join(f"{t}: {v:.2f}" for t, v in per.items()))

    pool = pseudo[(pseudo["n_dec"] >= 6) & ~pseudo.StudyInstanceUID.isin(labeled.StudyInstanceUID)]
    pool = pool.sort_values(["n_dec", "StudyInstanceUID"], ascending=[False, True])
    pseudo_selected = pool.head(CFG.pseudo_max).copy()
    print(f"pseudo: {len(pseudo_selected)} studies selected "
          f"({int(pseudo_selected['n_dec'].sum())} decisions) "
          f"of {len(pool)} eligible")
    TRAIN_UIDS = list(labeled.StudyInstanceUID) + list(pseudo_selected.StudyInstanceUID)
    tr_label = pd.concat([labeled, pseudo_selected[TARGETS + ["StudyInstanceUID"]]],
                         ignore_index=True)
    pseudo_w = {u: CFG.pseudo_weight for u in pseudo_selected.StudyInstanceUID}
else:
    pseudo_selected = pd.DataFrame()
    TRAIN_UIDS = list(labeled.StudyInstanceUID)
    tr_label = labeled
    pseudo_w = {}
print("train studies (labeled + pseudo):", len(TRAIN_UIDS))


## 2. EDA

Label prevalence, label correlations, series/plane/fluid composition, and a
DICOM header sample (transfer syntax, pixel spacing, photometric
interpretation) — helps you sanity-check the data and spot preprocessing
issues early (e.g. compressed DICOM that needs pylibjpeg).

In [ ]:
print("== label prevalence (labeled subset) ==")
prev = (labeled[TARGETS] == 1).mean().sort_values(ascending=False)
print(prev.round(3).to_string())
print()
print("== label correlation (Pearson, labeled subset) ==")
print(labeled[TARGETS].corr().round(2).to_string())
print()
print("== series composition (train) ==")
print("series per study:",
      train_ser.groupby("StudyInstanceUID").size().describe().round(2).to_string())
print()
print(train_ser.Anatomical_Plane.value_counts().to_string())
print()
print("plane x fluid-sensitive:")
print(pd.crosstab(train_ser.Anatomical_Plane, train_ser.Fluid_Sensitive).to_string())


In [ ]:
print("== DICOM header sample (up to 1200 random series, no pixel decode) ==")
rng = np.random.RandomState(0)
sids = rng.choice(train_ser.SeriesInstanceUID.unique(),
                  min(1200, train_ser.SeriesInstanceUID.nunique()), replace=False)
n_slices, tss, pis, sps, shapes = [], Counter(), Counter(), [], []
bad = 0
for sid in sids:
    row = train_ser[train_ser.SeriesInstanceUID == sid].iloc[0]
    d = os.path.join(CFG.train_dir, row.StudyInstanceUID, sid)
    files = glob.glob(os.path.join(d, "*.dcm"))
    n_slices.append(len(files))
    if not files:
        continue
    try:
        ds = pydicom.dcmread(files[0], stop_before_pixels=True)
        tss[str(ds.file_meta.TransferSyntaxUID)] += 1
        pis[str(getattr(ds, "PhotometricInterpretation", None))] += 1
        sp = getattr(ds, "PixelSpacing", None)
        sps.append((float(sp[0]), float(sp[1])) if sp and len(sp) >= 2 else (None, None))
        shapes.append((int(getattr(ds, "Rows", 0)), int(getattr(ds, "Columns", 0))))
    except Exception:
        bad += 1

n_slices = np.array(n_slices)
print("slices per series:", "min", n_slices.min(), "| median", int(np.median(n_slices)),
      "| max", n_slices.max(), "| >100 slices:", int((n_slices > 100).mean() * 100), "%")
print("unreadable sample files:", bad)
print()
print("transfer syntaxes:")
for ts, c in tss.most_common(6):
    comp = "COMPRESSED" if ts.startswith("1.2.840.10008.1.2.4") else "uncompressed"
    print(f"  {ts}  x{c}  [{comp}]")
print("photometric:", dict(pis))
sps = np.array([s for s in sps if s[0] is not None])
print("pixel spacing (mm), rows/cols of sample:",
      sps.min(0).round(3) if len(sps) else None, "to",
      sps.max(0).round(3) if len(sps) else None,
      "| shapes:", sorted(set(shapes))[:10])


## 3. Patient-aware cross-validation

A patient can appear in several studies, so naive study-level folds leak.
We group by `patient_id` (if present) with `StratifiedGroupKFold`, stratifying
on the label count. Falls back to `StratifiedKFold` / `KFold` automatically.

In [ ]:
PAT_COL = "patient_id" if "patient_id" in train_df.columns else None
print("patient column detected:", PAT_COL)
if PAT_COL:
    print("patients in train:", labeled[PAT_COL].nunique(),
          "| studies/patient:", labeled.groupby(PAT_COL).size().describe().round(2).to_string())
    multi = (labeled.groupby(PAT_COL).size() > 1).mean()
    print(f"patients with >1 study: {multi * 100:.1f}%  <-- why we group by patient")

y_sum = labeled[TARGETS].sum(axis=1)
try:
    bins = pd.qcut(y_sum, 4, labels=False, duplicates="drop")
except Exception:
    bins = None

if CFG.folds > len(labeled):
    CFG.folds = max(1, len(labeled))
    print(f"folds capped to {CFG.folds} (too few labeled studies)")

if CFG.folds < 2:
    # folds=1 quick test: patient-aware single holdout (80% train / 20% val)
    pat = labeled[PAT_COL].values if PAT_COL else np.arange(len(labeled))
    pats = np.unique(pat)
    n_tr = max(1, min(len(pats) - 1, int(len(pats) * 0.8)))
    tr_pats = set(np.random.RandomState(SEED).permutation(pats)[:n_tr])
    tr_idx = np.where(np.isin(pat, list(tr_pats)))[0]
    va_idx = np.where(~np.isin(pat, list(tr_pats)))[0]
    splits = [(tr_idx, va_idx)]
    print("using single holdout split (folds=1, patient-aware)")
elif PAT_COL and HAS_SGKF:
    try:
        groups = labeled[PAT_COL].values
        skf = StratifiedGroupKFold(n_splits=CFG.folds, shuffle=True, random_state=SEED)
        y_split = bins.values if bins is not None else np.zeros(len(labeled), dtype=int)
        splits = list(skf.split(labeled, y_split, groups=groups))
        print("using StratifiedGroupKFold (patient-aware)")
    except Exception as e:
        print("StratifiedGroupKFold failed:", e, "-> falling back")
        if bins is not None:
            splits = list(StratifiedKFold(n_splits=CFG.folds, shuffle=True,
                                          random_state=SEED).split(labeled, bins))
            print("using StratifiedKFold")
        else:
            splits = list(KFold(n_splits=CFG.folds, shuffle=True, random_state=SEED).split(labeled))
            print("using KFold")
elif bins is not None:
    splits = list(StratifiedKFold(n_splits=CFG.folds, shuffle=True,
                                  random_state=SEED).split(labeled, bins))
    print("using StratifiedKFold")
else:
    splits = list(KFold(n_splits=CFG.folds, shuffle=True, random_state=SEED).split(labeled))
    print("using KFold")

assert len(splits) == CFG.folds and len(splits[0][0]) > 0
for f, (tr, va) in enumerate(splits):
    print(f"  fold {f}: train studies={len(tr)} val studies={len(va)}")


## 4. Series selection

Knee MRI sequences ranked by diagnostic value:

1. **Sagittal fluid-sensitive** (PD/STIR/T2-FS): menisci, ACL, cartilage, effusion
2. **Coronal fluid-sensitive**: MCL, medial/lateral compartments, menisci
3. Other sagittal / coronal (T1, non-FS)
4. Axial (patellofemoral, patella)

We take up to `CFG.n_series` series per study following this preference order.
If a study has no series at all it is dropped from training and predicted 0.5
at test time.

In [ ]:
PLANE_PREF = ["Sagittal", "Coronal", "Axial"]


def pick_series(series_df, uid, n_series=CFG.n_series):
    rows = series_df[series_df.StudyInstanceUID == uid]
    if len(rows) == 0:
        return []
    chosen = []
    for plane in PLANE_PREF:
        sub = rows[rows.Anatomical_Plane == plane]
        if len(sub) == 0:
            continue
        fluid = sub[sub.Fluid_Sensitive == 1]
        sub = fluid if len(fluid) > 0 else sub
        sub = sub.sort_values("Fat_Suppression", ascending=False)
        chosen.append(sub.iloc[0].SeriesInstanceUID)
        if len(chosen) >= n_series:
            break
    return chosen


train_map = {u: pick_series(train_ser, u) for u in TRAIN_UIDS}
test_map = {u: pick_series(test_ser, u) for u in test_ser.StudyInstanceUID.unique()}

n_tr = sum(1 for v in train_map.values() if v)
n_te = sum(1 for v in test_map.values() if v)
print(f"train studies with >=1 series: {n_tr}/{len(train_map)}")
print(f"test  studies with >=1 series: {n_te}/{len(test_map)}")


def plane_stats(series_df, smap, name):
    chosen = [s for v in smap.values() for s in v]
    df = series_df[series_df.SeriesInstanceUID.isin(chosen)]
    print(f"{name}: chosen series per plane:")
    print(df.Anatomical_Plane.value_counts().to_string())
    print(df.Fluid_Sensitive.value_counts().rename("fluid").to_string())


plane_stats(train_ser, train_map, "train")
plane_stats(test_ser, test_map, "test")


## 5. Robust DICOM -> array + disk cache

`read_dicom_arr` applies, in order:

- `apply_voi_lut` if present (wrapped in try: some slices have none)
- MONOCHROME1 inversion (white-on-black -> black-on-white)
- multi-frame -> middle slice
- rescale slope/intercept (raw stored values -> physical MR units)

Then `process_series` windows each slice at percentiles 0.5–99.5, **resamples
by pixel spacing** to a common physical scale (`target_px_mm`), and resizes to
224x224. Results are cached as uint8 `.npy` (one file per series) so the epoch
loop and repeated runs stay fast. Unreadable slices (compressed DICOM without
codec) are skipped and counted.

In [ ]:
def read_dicom_arr(path):
    """-> (float32 2D array, pydicom Dataset). Applies VOI LUT, MONOCHROME1
    inversion, multi-frame handling and rescale slope/intercept."""
    ds = pydicom.dcmread(path, force=True)
    arr = ds.pixel_array
    arr = np.squeeze(np.asarray(arr, dtype=np.float32))
    try:
        from pydicom.pixel_data_handlers.util import apply_voi_lut
        arr = np.asarray(apply_voi_lut(arr, ds), dtype=np.float32)
    except Exception:
        pass
    if arr.ndim != 2:                    # multiframe -> keep middle slice
        arr = arr[arr.shape[0] // 2]
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = arr.max() - arr
    slope = float(getattr(ds, "RescaleSlope", 1) or 1)
    intercept = float(getattr(ds, "RescaleIntercept", 0) or 0)
    return arr * slope + intercept, ds


def normalize_window(arr, lo_p=0.5, hi_p=99.5):
    lo, hi = np.percentile(arr, [lo_p, hi_p])
    if hi <= lo:
        hi = lo + 1e-6
    arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    return (arr * 255).astype(np.uint8)


def resample_physical(arr, ds, target_mm=CFG.target_px_mm):
    """Resample so pixel spacing is ~target_mm on both axes (anatomy scale
    consistency across sites). Falls back to direct resize if no spacing."""
    sp = getattr(ds, "PixelSpacing", None)
    h, w = arr.shape[:2]
    if sp and len(sp) >= 2 and float(sp[0]) > 0 and float(sp[1]) > 0:
        new_h = int(round(h * float(sp[0]) / target_mm))
        new_w = int(round(w * float(sp[1]) / target_mm))
        new_h, new_w = max(64, min(new_h, 1600)), max(64, min(new_w, 1600))
        interp = cv2.INTER_CUBIC if (new_h, new_w) > (h, w) else cv2.INTER_AREA
        arr = cv2.resize(arr, (new_w, new_h), interpolation=interp)
    return arr


def process_series(series_dir, counter):
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    if not files:
        if counter:
            counter["empty_dir"] += 1
        return None

    def keyf(p):
        m = re.search(r"(\d+)", os.path.basename(p))
        return int(m.group(1)) if m else 10 ** 18

    files = sorted(files, key=keyf)
    if len(files) > CFG.max_slices_cache:
        idx = np.linspace(0, len(files) - 1, CFG.max_slices_cache).astype(int)
        files = [files[i] for i in idx]
    out = []
    for f in files:
        try:
            arr, ds = read_dicom_arr(f)
            arr = normalize_window(resample_physical(arr, ds))
            arr = cv2.resize(arr, (CFG.img_size, CFG.img_size), interpolation=cv2.INTER_AREA)
            out.append(arr)
        except Exception:
            if counter:
                counter["unreadable"] += 1
            continue
    if not out:
        if counter:
            counter["all_failed"] += 1
        return None
    return np.stack(out)                 # (S,H,W) uint8


def get_cached(uid, sid, base_dir, force=False):
    path = os.path.join(CFG.cache_dir, f"{uid}__{sid}.npy")
    if os.path.exists(path) and not force:
        return np.load(path)
    arr = process_series(os.path.join(base_dir, uid, sid), None)
    if arr is not None:
        np.save(path, arr)
    return arr


def build_cache(samples, base_dir):
    t0 = time.time()
    cnt = Counter()
    for i, (uid, sid) in enumerate(samples):
        path = os.path.join(CFG.cache_dir, f"{uid}__{sid}.npy")
        if not os.path.exists(path):
            arr = process_series(os.path.join(base_dir, uid, sid), cnt)
            if arr is not None:
                np.save(path, arr)
        if (i + 1) % 200 == 0:
            print(f"  cached {i+1}/{len(samples)}  ({time.time()-t0:.0f}s)")
    if cnt:
        print("  cache issues:", dict(cnt))


In [ ]:
train_samples = [(u, s) for u, sv in train_map.items() for s in sv]
test_samples  = [(u, s) for u, sv in test_map.items() for s in sv]
print("train samples:", len(train_samples), "| test samples:", len(test_samples))

# Cache version stamp: cached .npy are tied to img_size / max_slices / px_mm.
# If the config changed, stale caches would crash with shape errors - clear them.
os.makedirs(CFG.cache_dir, exist_ok=True)   # defensive: do not depend on cell order
CACHE_PARAMS = {"img_size": CFG.img_size, "max_slices": CFG.max_slices_cache,
                "px_mm": CFG.target_px_mm}
meta_path = os.path.join(CFG.cache_dir, "cache_meta.json")
old_meta = {}
if os.path.exists(meta_path):
    try:
        with open(meta_path, encoding="utf-8") as f:
            old_meta = json.load(f)
    except Exception:
        old_meta = {}
if old_meta != CACHE_PARAMS:
    cleared = 0
    for p in glob.glob(os.path.join(CFG.cache_dir, "*.npy")):
        os.remove(p)
        cleared += 1
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(CACHE_PARAMS, f)
    if cleared or not old_meta:
        print(f"cache prepared ({cleared} old files removed, config: {CACHE_PARAMS})")
else:
    print("cache version OK")

est_gb = ((len(train_samples) + len(test_samples)) * CFG.max_slices_cache
          * CFG.img_size * CFG.img_size / 1e9)
print(f"estimated cache size: {est_gb:.1f} GB "
      f"({CFG.max_slices_cache} slices x {CFG.img_size}^2 uint8 per series)")
if est_gb > 15:
    print("WARNING: cache may be large for /kaggle/working - if disk fills up,")
    print("         set CFG.max_slices_cache=32 and re-run this cell")

print("caching train series...")
build_cache(train_samples, CFG.train_dir)
print("caching test series...")
build_cache(test_samples, CFG.test_dir)

train_samples = [(u, s) for u, s in train_samples if get_cached(u, s, CFG.train_dir) is not None]
test_samples  = [(u, s) for u, s in test_samples  if get_cached(u, s, CFG.test_dir) is not None]
print("valid train samples:", len(train_samples), "| valid test samples:", len(test_samples))

cache_files = glob.glob(os.path.join(CFG.cache_dir, "*.npy"))
total_mb = sum(os.path.getsize(p) for p in cache_files) / 1e6
print(f"cache: {len(cache_files)} files, {total_mb:.0f} MB")


## 6. Text branch (radiology report)

Reports come in ~12 languages, so we avoid vocabularies: a **stateless
char-gram HashingVectorizer** (2–4 char n-grams, 512 dims, binary) is
multilingual by construction and needs no pretrained model — perfect for the
offline constraint. The hashed vector feeds an MLP that is concatenated with
CNN features before the final head.

If the test set has no reports (`test.csv` absent or no `Report` column),
the test text is set to zeros: the model still works, image-only. Set
`CFG.use_text=False` for a pure-vision model.

In [ ]:
TEXT_DIM = 512
TEXT_ON = CFG.use_text and "Report" in train_df.columns
print("text branch:", "ON" if TEXT_ON else "OFF")

if TEXT_ON:
    from sklearn.feature_extraction.text import HashingVectorizer
    vec = HashingVectorizer(n_features=TEXT_DIM, analyzer="char_wb",
                            ngram_range=(2, 4), binary=True, norm="l2",
                            alternate_sign=False)

    def build_text_lookup(df):
        rep = df[["StudyInstanceUID", "Report"]].copy()
        rep["Report"] = rep["Report"].fillna("").astype(str)
        X = vec.transform(rep["Report"].values).toarray().astype(np.float32)
        return {uid: X[i] for i, uid in enumerate(rep.StudyInstanceUID)}

    text_lookup = build_text_lookup(train_df)
    print(f"train text features: {len(text_lookup)} studies x {TEXT_DIM}")

    text_lookup_test = None
    test_csv = os.path.join(CFG.data_dir, "test.csv")
    if os.path.exists(test_csv):
        tdf = pd.read_csv(test_csv)
        if "Report" in tdf.columns:
            text_lookup_test = build_text_lookup(tdf)
            print(f"test reports found: {len(text_lookup_test)} studies (real text)")
        else:
            print("test.csv exists but has no Report -> zeros at inference")
    else:
        print("no test.csv -> zeros at inference")
else:
    text_lookup, text_lookup_test = None, None


def text_row(uid, lookup, dim=TEXT_DIM):
    v = lookup.get(uid) if lookup else None
    return v if v is not None else np.zeros(dim, dtype=np.float32)


## 7. Dataset

Each sample = one (study, series) stack. A fixed window of `CFG.n_slices`
slices is drawn: random window in training, sliding window (`offset`) in
validation/TTA. Augmentations (horizontal flip, small rotation, brightness
+ contrast jitter) are applied per slice; padded slots are masked out so
attention pooling ignores them.

In [ ]:
class KneeDataset(Dataset):
    def __init__(self, samples, base_dir, text_lookup=None, train_mode=True, offset=0):
        self.samples = samples
        self.base_dir = base_dir
        self.text_lookup = text_lookup
        self.train_mode = train_mode
        self.offset = offset

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        uid, sid = self.samples[i]
        arr = get_cached(uid, sid, self.base_dir)      # (S,H,W) uint8
        S, K, H, W = len(arr), CFG.n_slices, CFG.img_size, CFG.img_size
        if self.train_mode and S > K:
            start = np.random.randint(0, S - K + 1)
            idx = np.arange(start, start + K)
        elif S >= K:
            step = max(1, S - K + 1)
            start = min(self.offset, step - 1)
            idx = np.arange(start, start + K)
        else:
            idx = np.linspace(0, S - 1, K).astype(int)

        real = min(S, K)
        imgs = np.zeros((K, H, W), dtype=np.float32)
        imgs[:real] = arr[idx[:real]].astype(np.float32) / 255.0
        for k in range(real):
            img = imgs[k]
            if self.train_mode:
                if random.random() < 0.5:
                    img = img[:, ::-1]
                angle = random.uniform(-7, 7)
                if abs(angle) > 0.1:
                    M = cv2.getRotationMatrix2D((W / 2, H / 2), angle, 1.0)
                    img = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR,
                                         borderMode=cv2.BORDER_REPLICATE)
                img = np.clip(img * random.uniform(0.9, 1.1) + random.uniform(-0.04, 0.04), 0, 1)
            imgs[k] = (img - IMAGENET_MEAN[0]) / IMAGENET_STD[0]

        mask = np.zeros(K, dtype=np.bool_)
        mask[:real] = True
        text = text_row(uid, self.text_lookup)
        return (torch.from_numpy(imgs).float(), torch.from_numpy(mask),
                torch.from_numpy(text), i)


## 8. Model

- Backbone: ImageNet-pretrained 2D CNN (DenseNet-121 default, EfficientNet-B0
  option). Weights must be attached for offline scoring — see helper cell at
  the end; without them the model falls back to random init.
- **Slice attention**: each slice's feature vector gets an attention score
  (small Linear); masked slots are set to -inf; softmax over valid slices;
  weighted average -> series feature.
- **Text MLP**: hashed report vector -> ReLU(128) -> series feature concat.
- Head: dropout 0.2 + Linear(D + 512 or D, 12).

**Offline note:** torchvision cannot download weights at scoring time. Save
them during development (`CFG.save_weights = True`, last cell), attach as a
dataset, set `CFG.pretrained_path`.

In [ ]:
def load_backbone(backbone):
    pretrained = bool(CFG.pretrained_path) and os.path.exists(CFG.pretrained_path)
    if backbone == "densenet121":
        from torchvision.models import densenet121, DenseNet121_Weights
        if not pretrained:
            try:
                base = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
            except Exception as e:
                print("WARNING: ImageNet download failed -> random init:", e)
                base = densenet121(weights=None)
        else:
            base = densenet121(weights=None)
        D = 1024
    elif backbone == "efficientnet_b0":
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        if not pretrained:
            try:
                base = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
            except Exception as e:
                print("WARNING: ImageNet download failed -> random init:", e)
                base = efficientnet_b0(weights=None)
        else:
            base = efficientnet_b0(weights=None)
        D = 1280
    else:
        raise ValueError(f"unknown backbone: {backbone}")
    if pretrained:
        sd = torch.load(CFG.pretrained_path, map_location="cpu")
        sd = {k: v for k, v in sd.items() if k in base.state_dict()}
        base.load_state_dict(sd, strict=False)
        print("loaded attached pretrained weights")
    return base, D


class KneeModel(nn.Module):
    def __init__(self, backbone="densenet121", num_classes=12, text_dim=0):
        super().__init__()
        base, D = load_backbone(backbone)
        self.features = base.features
        self.text_dim = text_dim
        self.attn = nn.Linear(D, 1)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(D + text_dim, num_classes))

    def forward(self, x, mask, text=None):
        # x: (B,K,H,W)  mask: (B,K)  text: (B,TEXT_DIM) or None
        # Frames are run through the backbone in chunks of CFG.frames_chunk so
        # GPU memory stays bounded regardless of batch size / series length.
        B, K = x.shape[:2]
        flat = x.view(-1, 1, x.size(2), x.size(3))
        feats = []
        for c in range(0, len(flat), CFG.frames_chunk):
            x3 = flat[c:c + CFG.frames_chunk].expand(-1, 3, -1, -1)
            f = self.features(x3)
            feats.append(F.adaptive_avg_pool2d(f, 1).flatten(1))
        feats = torch.cat(feats, 0).view(B, K, -1)
        att = self.attn(feats).squeeze(-1)
        att = att.masked_fill(~mask, float("-inf"))
        att = F.softmax(att, dim=1).unsqueeze(-1)
        pooled = (feats * att).sum(1)                 # (B, D)
        if self.text_dim and text is not None:
            pooled = torch.cat([pooled, text], dim=1)
        return self.head(pooled)


class EMA:
    """Exponential moving average of model weights (evaluated, not trained)."""

    def __init__(self, model, decay=0.99):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype != torch.float32:     # e.g. BN num_batches_tracked (long)
                self.shadow[k].copy_(v.detach())
                continue
            self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)

    def swap(self, model):
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)
        return backup

    def restore(self, model, backup):
        model.load_state_dict(backup)


def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, train


## 9. Training loop

- Loss: BCE + 0.02 label smoothing on study-level logits
- AdamW (head 1e-3 / body 3e-5), linear warmup (5%) + cosine decay
- Epoch 1: head-only (backbone frozen); epochs 2+: full fine-tune
- AMP on GPU, gradient clipping at 1.0, EMA decay 0.99 (validation uses EMA)
- Early stopping on val macro AUC (patience), best EMA state per fold saved
- Per-label AUC printed each epoch for debugging weak classes


In [ ]:
def sigmoid(z):
    """Stable sigmoid probabilities from logits (clip guards exp overflow)."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


def macro_auc(y_df, probs):
    scores = []
    for j, t in enumerate(TARGETS):
        try:
            s = roc_auc_score(y_df[t].values, probs[:, j])
        except ValueError:
            continue
        if np.isfinite(s):                 # sklearn may return NaN for single-class y
            scores.append(s)
    return float(np.mean(scores)) if scores else 0.0


def per_label_auc(y_df, probs):
    out = {}
    for j, t in enumerate(TARGETS):
        if y_df[t].nunique() > 1:
            try:
                s = roc_auc_score(y_df[t].values, probs[:, j])
            except ValueError:
                continue
            if np.isfinite(s):
                out[t] = round(s, 4)
    return out


def predict_studies(model, samples, gt_ids, base_dir, text_lookup=None, grids=CFG.tta_grids):
    """Study-level logits (N,12). grids>1 slides the slice window (TTA)."""
    model.eval()
    pos = {u: i for i, u in enumerate(gt_ids)}
    sums = np.zeros((len(gt_ids), len(TARGETS)))
    cnts = np.zeros(len(gt_ids))
    for g in range(grids):
        offset = int(round(g * CFG.n_slices / max(grids - 1, 1)))
        ds = KneeDataset(samples, base_dir, text_lookup=text_lookup,
                         train_mode=False, offset=offset)
        dl = DataLoader(ds, batch_size=CFG.batch_size, shuffle=False, num_workers=0)
        with torch.no_grad():
            for imgs, mask, text, idxs in dl:
                imgs, mask = imgs.to(DEVICE), mask.to(DEVICE)
                text = text.to(DEVICE) if model.text_dim else None
                B = imgs.size(0)
                logits = model(imgs, mask, text)
                for b in range(B):
                    p = pos[samples[idxs[b]][0]]
                    sums[p] += logits[b].cpu().numpy()
                    cnts[p] += 1
    return sums / np.maximum(cnts, 1)[:, None]


def train_fold(fold, tr_samples, va_samples, va_ids, label_df, text_lookup, w_map=None):
    seed_everything(SEED + fold)
    model = KneeModel(backbone=CFG.backbone, num_classes=len(TARGETS),
                      text_dim=TEXT_DIM if TEXT_ON else 0)
    if CFG.use_compile and torch.cuda.is_available():
        try:
            model = torch.compile(model)
        except Exception:
            pass
    model = model.to(DEVICE)
    total_p, train_p = count_params(model)
    print(f"  params: total={total_p/1e6:.1f}M trainable={train_p/1e6:.1f}M")

    tr_ids = list(dict.fromkeys(u for u, _ in tr_samples))
    label_arr = label_df.set_index("StudyInstanceUID").reindex(tr_ids)
    labs = label_arr[TARGETS].values.astype(np.float32)
    # pseudo-label rows can be NaN on undecided targets -> mask from the loss
    lab_mask = ~np.isnan(labs)
    labs = np.nan_to_num(labs)
    labels = torch.tensor(labs, dtype=torch.float32, device=DEVICE)
    label_mask = torch.tensor(lab_mask, dtype=torch.bool, device=DEVICE)
    if w_map:
        w = torch.tensor([w_map.get(u, 1.0) for u in tr_ids], dtype=torch.float32,
                         device=DEVICE)
    else:
        w = torch.ones(len(tr_ids), dtype=torch.float32, device=DEVICE)
    pos_of_uid = {u: i for i, u in enumerate(tr_ids)}

    head_params, body_params = [], []
    for name, p in model.named_parameters():
        if name.startswith("classifier") or name.startswith("head") or name.startswith("attn"):
            head_params.append(p)
        else:
            body_params.append(p)

    optimizer = torch.optim.AdamW(
        [{"params": body_params, "lr": CFG.lr_body},
         {"params": head_params, "lr": CFG.lr_head}],
        weight_decay=CFG.weight_decay)

    total_steps = CFG.epochs * ((len(tr_samples) + CFG.batch_size - 1) // CFG.batch_size)
    warmup_steps = int(total_steps * CFG.warmup_frac)

    def lr_lambda(step):
        if step < warmup_steps:
            return 0.1 + 0.9 * step / max(1, warmup_steps)
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * t))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.use_amp)

    def bce_loss(logits, targets, wgt=None, msk=None):
        # BCEWithLogits with manual label smoothing (the label_smoothing kwarg
        # is not supported on every torch version); masked+weighted so pseudo
        # rows (NaN targets) and low-weight pseudo labels are handled
        y = targets * (1.0 - CFG.label_smoothing) + 0.5 * CFG.label_smoothing
        loss = F.binary_cross_entropy_with_logits(logits, y, reduction="none")
        if wgt is not None:
            loss = loss * wgt[:, None]
        if msk is not None:
            m = msk.float()
            loss = loss * m
        else:
            m = torch.ones_like(loss)
        return loss.sum() / m.sum().clamp_min(1.0)

    ema = EMA(model, CFG.ema_decay)

    best_auc, best_ema_state, no_improve = -1.0, None, 0

    for ep in range(1, CFG.epochs + 1):
        freeze = ep <= CFG.freeze_epochs
        for p in body_params:
            p.requires_grad_(not freeze)
        ds_tr = KneeDataset(tr_samples, CFG.train_dir,
                            text_lookup=text_lookup, train_mode=True)
        dl_tr = DataLoader(ds_tr, batch_size=CFG.batch_size, shuffle=True,
                           num_workers=CFG.num_workers, pin_memory=(DEVICE == "cuda"))
        model.train()
        t0 = time.time()
        for step, (imgs, mask, text, idxs) in enumerate(dl_tr):
            imgs, mask = imgs.to(DEVICE), mask.to(DEVICE)
            text = text.to(DEVICE) if model.text_dim else None
            B = imgs.size(0)
            with torch.autocast("cuda", enabled=CFG.use_amp):
                logits = model(imgs, mask, text)          # (B,12) per sample
                # per-study logits indexed by study position in tr_ids
                # (tensor sized len(tr_ids), NOT batch size: batch samples can
                # belong to studies with position > B)
                study_logits = torch.zeros(len(tr_ids), len(TARGETS), device=DEVICE)
                counts = torch.zeros(len(tr_ids), device=DEVICE)
                for b in range(B):
                    p = pos_of_uid[tr_samples[idxs[b]][0]]
                    study_logits[p] += logits[b]
                    counts[p] += 1
                # loss only over studies present in THIS batch
                sel = torch.nonzero(counts > 0).squeeze(1)
                if len(sel) == 0:
                    continue
                study_logits = study_logits[sel] / counts[sel].clamp_min(1).unsqueeze(1)
                loss = bce_loss(study_logits, labels[sel], w[sel], label_mask[sel])
            scaler.scale(loss).backward()
            if CFG.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            ema.update(model)
            if step % 50 == 0:
                print(f"  fold{fold} ep{ep} step{step}/{len(dl_tr)} loss={loss.item():.4f}")
        del dl_tr

        backup = ema.swap(model)
        va_logits = predict_studies(model, va_samples, va_ids, CFG.train_dir,
                                    text_lookup, grids=1)
        ema.restore(model, backup)
        va_prob = sigmoid(va_logits)
        auc = macro_auc(label_df.set_index("StudyInstanceUID").loc[va_ids][TARGETS], va_prob)
        pla = per_label_auc(label_df.set_index("StudyInstanceUID").loc[va_ids][TARGETS], va_prob)
        print(f"  fold{fold} ep{ep} val_macro_auc={auc:.4f}  ({time.time()-t0:.0f}s/epoch)")
        print("  per-label:", " ".join(f"{k}:{v:.4f}" for k, v in pla.items()))
        if auc > best_auc + 1e-4:
            best_auc, no_improve = auc, 0
            best_ema_state = {k: v.detach().cpu().clone() for k, v in ema.shadow.items()}
        else:
            no_improve += 1
            if no_improve >= CFG.patience:
                print(f"  early stop at epoch {ep}")
                break

    if best_ema_state is None:             # no improvement at all -> keep EMA
        best_ema_state = {k: v.detach().cpu().clone() for k, v in ema.shadow.items()}
    torch.save(best_ema_state, f"{CFG.out_dir}/knee_fold{fold}.pt")
    return best_auc


In [ ]:
results = []
for fold, (tr_idx, va_idx) in enumerate(splits):
    if elapsed_hours() > CFG.budget_hours:
        print(f"[budget] {elapsed_hours():.1f}h elapsed > {CFG.budget_hours}h")
        print("  -> skipping remaining folds")
        break
    tr_u = set(labeled.iloc[tr_idx].StudyInstanceUID)
    if CFG.use_pseudo and pseudo_w:
        # pseudo studies of this fold: all except val studies
        tr_u |= set(pseudo_selected.StudyInstanceUID) - set(labeled.iloc[va_idx].StudyInstanceUID)
    va_u = list(labeled.iloc[va_idx].StudyInstanceUID)
    tr_samples = [(u, s) for u, s in train_samples if u in tr_u]
    va_samples = [(u, s) for u, s in train_samples if u in va_u]
    print(f"== fold {fold}: train studies={len(tr_u)} val studies={len(va_u)} "
          f"(elapsed {elapsed_hours():.2f}h) ==")
    t0 = time.time()
    auc = train_fold(fold, tr_samples, va_samples, va_u, tr_label, text_lookup, pseudo_w)
    results.append(auc)
    print(f"== fold {fold} done, val macro AUC={auc:.4f} ({time.time()-t0:.0f}s) ==")
    free_memory()

if results:
    print("fold AUCs:", [round(a, 4) for a in results],
          "-> mean", round(float(np.mean(results)), 4))
else:
    print("WARNING: no fold completed (budget/subset too small)")


## 10. Inference -> submission.csv

Ensemble over trained folds (mean of sigmoid probabilities) with sliding-window
TTA. The final submission is merged against `sample_submission` so every test
row is present; studies without usable data default to 0.5.

In [ ]:
test_ids = sorted(set(u for u, _ in test_samples) | set(test_map.keys()))
print(f"test studies: {len(test_ids)} | test samples: {len(test_samples)}")
if len(test_samples) == 0:
    print("WARNING: no valid test series -> submission will be all 0.5")

grids = CFG.tta_grids if elapsed_hours() < CFG.budget_hours else 1
all_probs = np.zeros((len(test_ids), len(TARGETS)))
used = 0
for fold in range(CFG.folds):
    ckpt = os.path.join(CFG.out_dir, f"knee_fold{fold}.pt")
    if not os.path.exists(ckpt):
        continue
    model = KneeModel(backbone=CFG.backbone, num_classes=len(TARGETS),
                      text_dim=TEXT_DIM if TEXT_ON else 0).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    logits = predict_studies(model, test_samples, test_ids, CFG.test_dir,
                             text_lookup_test, grids=grids)
    all_probs += sigmoid(logits)
    used += 1
    del model
    free_memory()
    print(f"fold {fold} inference done (elapsed {elapsed_hours():.2f}h)")

if used == 0:
    print("WARNING: no fold checkpoints found -> submission will be all 0.5")
    all_probs = np.full((len(test_ids), len(TARGETS)), 0.5)
else:
    all_probs /= used
    print(f"ensembled {used} fold(s), grids={grids}")


In [ ]:
prob_df = pd.DataFrame(all_probs, columns=TARGETS)
prob_df["StudyInstanceUID"] = test_ids
sub = sample_sub[["StudyInstanceUID"]].merge(prob_df, on="StudyInstanceUID", how="left").fillna(0.5)
sub = sub[sample_sub.columns]

assert len(sub) == len(sample_sub)
assert list(sub.columns) == list(sample_sub.columns)
assert sub.isna().sum().sum() == 0
assert (sub[TARGETS] >= 0).all().all() and (sub[TARGETS] <= 1).all().all()

sub.to_csv(f"{CFG.out_dir}/submission.csv", index=False)
print("saved /kaggle/working/submission.csv | rows:", len(sub))
print(sub.head())
print()
print("predicted probability stats (sanity: no NaN/constant columns):")
print(sub[TARGETS].agg(["mean", "min", "max"]).round(3).to_string())


## 11. Offline-weights helper (dev only) + timing

Set `CFG.save_weights = True` in the config cell and run the notebook **once
during development (internet ON)** — the last cell dumps the ImageNet backbone
to `/kaggle/working`. Then create a dataset from that file (Output tab ->
Create dataset, or download & upload), attach it to this notebook and set
`CFG.pretrained_path` to e.g.
`/kaggle/input/knee-backbone/densenet121_backbone.pth` (set
`CFG.save_weights = False` again).

Also useful for the **Efficiency award**: it is scored as
`AUC/(Benchmark-maxAUC) + RuntimeSeconds/32400`, i.e. faster is better —
caching, no heavy TTA and small ensembles all help. Check the elapsed time
below and tighten `CFG` if needed.

In [ ]:
# ---- DEV ONLY: save backbone weights for offline scoring ----
if CFG.save_weights:
    base, _ = load_backbone(CFG.backbone)
    torch.save(base.state_dict(), f"{CFG.out_dir}/{CFG.backbone}_backbone.pth")
    print("saved ->", f"{CFG.out_dir}/{CFG.backbone}_backbone.pth")

print(f"total runtime: {elapsed_hours():.2f} h")
print("submission ready: /kaggle/working/submission.csv")
